# ProductIQ Phase 3 — Step 3.3: Structured Product Representation

Build typed `ProductRepresentation` objects from the Phase 2 processed catalog (read-only).

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

from data.export.schema import PROCESSED_COLUMN_ORDER, PROCESSED_PARQUET_FILENAME
from productiq.representation import (
    build_product_representation,
    build_product_representations,
    representation_to_dict,
    representation_to_json,
    validate_product_representation,
)


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    msg = "Could not locate ProductIQ project root"
    raise RuntimeError(msg)


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
PARQUET_PATH = PROJECT_ROOT / "resources" / "processed" / PROCESSED_PARQUET_FILENAME
SAMPLE_SIZE = 500

catalog = pd.read_parquet(PARQUET_PATH, engine="pyarrow")
print("Rows:", len(catalog), "Columns:", len(catalog.columns))
print("Path:", PARQUET_PATH)

## 1. Input schema (Phase 2 contract)

In [ ]:
print(list(PROCESSED_COLUMN_ORDER))
catalog.dtypes

## 2. Representative products

In [ ]:
sample = catalog.head(SAMPLE_SIZE)
records = sample.to_dict(orient="records")
representations = build_product_representations(records)
print("Built:", len(representations))
representations[0]

## 3. Scalar attributes

In [ ]:
rep = representations[0]
print("product_id:", rep.product_id)
print("brand:", rep.brand)
print("category_gender:", rep.category_gender)
print("product_type:", rep.product_type)

## 4. Multi-value parsing (pipe-delimited → lists)

In [ ]:
multi_cols = [
    "material",
    "pattern",
    "product_features",
    "style_attributes",
]
pipe_rows = sample[sample[multi_cols].astype(str).str.contains("\\|", regex=True, na=False).any(axis=1)]
if len(pipe_rows):
    row = pipe_rows.iloc[0].to_dict()
    parsed = build_product_representation(row)
    print("Source material:", row.get("material"))
    print("Representation material:", parsed.material)
else:
    print("No pipe-delimited row in sample head; check broader sample.")

## 5. Missing values (`null`, not invented defaults)

In [ ]:
sparse = sample[sample["material"].isna() | (sample["material"] == "")].head(3)
for _, row in sparse.iterrows():
    rep = build_product_representation(row.to_dict())
    print(rep.product_id, "material=", rep.material)

## 6. Serialized dict / JSON

In [ ]:
payload = representation_to_dict(representations[0])
print(json.dumps(payload, indent=2)[:1200])
print("---")
print(representation_to_json(representations[0])[:400])

## 7. Validation

In [ ]:
for rep in representations[:10]:
    validate_product_representation(rep)
print("Validated 10 representative representations.")

## 8. Batch conversion summary

In [ ]:
source_ids = set(sample["product_id"])
built_ids = {rep.product_id for rep in representations}
assert source_ids == built_ids
assert len(representations) == len(sample)

non_null_counts = {
    "product_type": sum(1 for r in representations if r.product_type),
    "color": sum(1 for r in representations if r.color),
    "material": sum(1 for r in representations if r.material),
    "pattern": sum(1 for r in representations if r.pattern),
}
print("Batch size:", len(representations))
print("Non-null field counts:", non_null_counts)
print("product_id preservation: OK")
print("Row count preservation: OK")